In [2]:
import polars as pl

from genpp.eval import best_models

In [3]:
res = []
for model_class, models in best_models:
    for model in models:
        model_scores_file = model.model_dir / "scores.csv"
        model_scores_df = pl.read_csv(model_scores_file)
        tag = model.tag or ""
        model_scores_df = model_scores_df.with_columns(pl.lit(tag).alias("tag")).with_columns(
            pl.col("Metric").replace("EnsembleCRPS", "CRPS")
        )
        res.append(model_scores_df)
scores_df = pl.concat(res)

In [4]:
scores_df.filter(pl.col("Variable") == "combined").pivot(
    on="Metric", index=["Model", "tag"], values="Score"
)

Model,tag,VariogramScore,CRPS,EnergyScore,PatchwiseEnergyScore,MultiScaleEnergyScore,MultiScalePatchwiseEnergyScore
str,str,f64,f64,f64,f64,f64,f64
"""EMOS+ECC""","""""",476588.8125,0.60354,38.260643,0.714378,0.702779,0.630238
"""EMOS+GCA""","""""",485930.0625,0.61495,38.688656,0.723106,0.711365,0.639297
"""DRNModel+ECC""","""""",482516.59375,0.59231,37.363914,0.70116,0.684577,0.615225
"""DRNModel+GCA""","""""",488374.65625,0.60205,37.676483,0.707426,0.691716,0.623377
"""CNNChenNoiseModel""","""ind_es""",489685.5,0.632431,38.164589,0.735939,0.736328,0.677911
…,…,…,…,…,…,…,…
"""CNNEngressionNoiseModel""","""ind_mspes""",485190.40625,0.608088,37.949768,0.711632,0.702607,0.631569
"""CNNEngressionDirectModel""","""dir_es""",484443.78125,0.623556,37.659119,0.725163,0.711159,0.658415
"""CNNEngressionDirectModel""","""dir_pes""",477042.03125,0.60223,37.469082,0.704365,0.693311,0.624883


In [ ]:
# Create a LaTeX table from scores_df, similar style to 02plot_scores
score_df = (
    scores_df.filter(pl.col("Variable") == "combined")
    .select(["Model", "tag", "Metric", "Score"])
    .with_columns(
        pl.col("Model")
        .replace(
            {
                "CNNChenNoiseModel": "LNGM",
                "CNNChenDirectModel": "LNGM",
                "CNNEngressionNoiseModel": "ENG",
                "CNNEngressionDirectModel": "ENG",
            }
        )
        .str.replace("_", "\\_", literal=True)
        .alias("Model"),
        pl.col("tag").alias("Tag"),
        pl.when(pl.col("tag").str.starts_with("dir"))
        .then(pl.lit("direct"))
        .when(pl.col("tag").str.starts_with("ind"))
        .then(pl.lit("indirect"))
        .otherwise(pl.lit(""))
        .alias("Directness"),
        pl.when(pl.col("tag").is_not_null())
        .then(
            pl.col("tag")
            .str.replace(r"^(?:dir_|ind_)", "", literal=False)
            .str.replace("_", "", literal=False)
            .str.to_uppercase()
        )
        .otherwise(pl.lit(""))
        .alias("Loss Fn"),
    )
    .pivot(on="Metric", index=["Model", "Loss Fn", "Directness"], values="Score")
)

# enforce required column order and abbreviations
ordered_cols = [
    "CRPS",
    "EnergyScore",
    "PatchwiseEnergyScore",
    "MultiScaleEnergyScore",
    "MultiScalePatchwiseEnergyScore",
    "VariogramScore",
]
score_cols = [col for col in ordered_cols if col in score_df.columns]

best = {}
second_best = {}
for col in score_cols:
    sorted_vals = score_df[col].drop_nulls().sort().to_list()
    if len(sorted_vals) > 0:
        best[col] = sorted_vals[0]
        second_best[col] = sorted_vals[1] if len(sorted_vals) > 1 else None

latex_lines = []
latex_lines.append(r"\begin{table}[ht]")
latex_lines.append(r"\centering")
latex_lines.append(r"\begin{tabular}{lllc" + "r" * len(score_cols) + "}")
latex_lines.append(r"\toprule")
# Column labels with scale factors in labels
col_labels = {
    "CRPS": r"\textbf{CRPS} ($\times 10^{-1}$)",
    "EnergyScore": r"\textbf{ES} ($\times 10^{1}$)",
    "PatchwiseEnergyScore": r"\textbf{PES} ($\times 10^{-1}$)",
    "MultiScaleEnergyScore": r"\textbf{MSES} ($\times 10^{-1}$)",
    "MultiScalePatchwiseEnergyScore": r"\textbf{MSPES} ($\times 10^{-1}$)",
    "VariogramScore": r"\textbf{VS} ($\times 10^{5}$)",
}
scale_factor = {
    "CRPS": 1e-1,
    "EnergyScore": 1e1,
    "PatchwiseEnergyScore": 1e-1,
    "MultiScaleEnergyScore": 1e-1,
    "MultiScalePatchwiseEnergyScore": 1e-1,
    "VariogramScore": 1e5,
}
header = [r"\textbf{Model}", r"\textbf{Loss Fn}", r"\textbf{Direct?}"] + [
    col_labels.get(col, f"\textbf{{{col}}}") for col in score_cols
]
latex_lines.append(" & ".join(header) + r" \\")
latex_lines.append(r"\midrule")

for row in score_df.iter_rows(named=True):
    direct = row.get("Directness", "")
    if direct == "direct":
        direct_mark = r"\\ding{51}"
    elif direct == "indirect":
        direct_mark = r"\\ding{55}"
    else:
        direct_mark = "--"

    cells = [row["Model"], row.get("Loss Fn", ""), direct_mark]
    for col in score_cols:
        v = row[col]
        if v is None:
            formatted = "--"
        else:
            factor = scale_factor.get(col, 1.0)
            formatted_val = v / factor
            formatted = f"{formatted_val:.2f}"
            if v == best.get(col):
                formatted = r"\textbf{" + formatted + r"}"
            elif v == second_best.get(col):
                formatted = r"\underline{" + formatted + r"}"
        cells.append(formatted)
    latex_lines.append(" & ".join(cells) + r" \\")

latex_lines.append(r"\bottomrule")
latex_lines.append(r"\end{tabular}")
latex_lines.append(r"\caption{Scores table for combined variable}")
latex_lines.append(r"\label{tab:scores_table}")
latex_lines.append(r"\end{table}")

latex_str = "\n".join(latex_lines)
print(latex_str)

\begin{table}[ht]
\centering
\begin{tabular}{lllcrrrrrr}
\toprule
\textbf{Model} & \textbf{Loss Fn} & \textbf{Direct?} & \textbf{CRPS} ($\times 10^{-1}$) & \textbf{ES} ($\times 10^{1}$) & \textbf{PES} ($\times 10^{1}$) & \textbf{MSES} ($\times 10^{1}$) & \textbf{MSPES} ($\times 10^{1}$) & \textbf{VS} ($\times 10^{5}$) \\
\midrule
EMOS+ECC &  & -- & 6.04 & 3.83 & 7.14 & 7.03 & 6.30 & 4.77 \\
EMOS+GCA &  & -- & 6.15 & 3.87 & 7.23 & 7.11 & 6.39 & 4.86 \\
DRNModel+ECC &  & -- & \underline{5.92} & 3.74 & 7.01 & \underline{6.85} & \underline{6.15} & 4.83 \\
DRNModel+GCA &  & -- & 6.02 & 3.77 & 7.07 & 6.92 & 6.23 & 4.88 \\
LNGM & ES & \\ding{55} & 6.32 & 3.82 & 7.36 & 7.36 & 6.78 & 4.90 \\
LNGM & PES & \\ding{55} & 6.00 & 3.73 & 7.02 & 6.94 & 6.26 & 4.76 \\
LNGM & MSES & \\ding{55} & 6.50 & 3.79 & 7.35 & 7.01 & 6.54 & 4.93 \\
LNGM & MSPES & \\ding{55} & 6.07 & 3.78 & 7.09 & 7.00 & 6.31 & 4.85 \\
LNGM & ES & \\ding{51} & 6.23 & 3.78 & 7.28 & 7.24 & 6.66 & 4.93 \\
LNGM & PES & \\ding{51} & 6.01